# Setup

In [1]:
# transformer-lens 3.4.0 pins torchvision>=0.22,<0.23 (for torch 2.7.x),
# but Colab has torch 2.10 + torchvision 0.25. This breaks pip's resolver.
# Workaround: install both with --no-deps, then install dependencies separately.

# Step 1: Install sae-lens and transformer-lens without the torchvision constraint
!pip install --no-deps transformer-lens sae-lens

# Step 2: Install all actual dependencies (minus torchvision, not needed for text SAEs)
!pip install \
    "transformers>=5.4.0,<6.0.0" \
    "datasets>=3.1.0" \
    "einops>=0.6.0" \
    "jaxtyping>=0.2.11" \
    "fancy-einsum>=0.0.3" \
    "beartype>=0.14.1" \
    "accelerate>=0.23.0" \
    "rich>=12.6.0" \
    "sentencepiece" \
    "typeguard>=4.2,<5" \
    "wandb>=0.13.5" \
    "transformers-stream-generator>=0.0.5,<0.1" \
    "better-abc>=0.0.3" \
    "protobuf>=3.20.0" \
    "huggingface-hub>=0.23.2" \
    "safetensors>=0.4.2,<1.0.0" \
    "plotly>=5.19.0" \
    "nltk>=3.8.1" \
    "python-dotenv>=1.0.1" \
    "pyyaml>=6.0.1" \
    "simple-parsing>=0.1.8" \
    "tenacity>=9.0.0" \
    "babe>=0.0.7"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 37.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 16.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.3/276.3 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 30.1 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=421a6ccb392b75403afbf19aa413f1aec499fc4dafa5b18238aca9a4493734d3
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3

In [ ]:
from huggingface_hub import login
import os
from __future__ import annotations

from tqdm import tqdm
import pandas as pd
import argparse
import re
import sys
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable
from torch.utils.data import DataLoader

import requests
import torch, gc
from sae_lens import SAE
from transformer_lens import HookedTransformer

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive"
REPO_DIR = "/content/SAE_Surrogate"

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))
print(torch.cuda.mem_get_info())
print(torch.cuda.is_bf16_supported())

Mounted at /content/drive
True
0
NVIDIA A100-SXM4-40GB
(41957130240, 42405855232)
True


In [4]:
with open(f"{DRIVE_DIR}/tokens/hftoken.txt") as f:
    token = f.readline().strip()

login(token=token)

In [5]:
with open(f"{DRIVE_DIR}/tokens/ghtoken.txt") as f:
    email = f.readline().strip()
    username = f.readline().strip()
    token = f.readline().strip()

!git config --global user.email "{email}"
!git config --global user.name "{username}"

#!git clone https://github.com/ntjohn04/SAE_Surrogate
if not os.path.exists(REPO_DIR):
    !git clone https://{token}@github.com/{username}/SAE_Surrogate
else:
    !git -C {REPO_DIR} pull

sys.path.append(f"{REPO_DIR}")

import util_neuronpedia
import util_nanogpt
import util_GSM8k

Cloning into 'SAE_Surrogate'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 61 (delta 31), reused 45 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 428.31 KiB | 20.40 MiB/s, done.
Resolving deltas: 100% (31/31), done.


# Load Models

In [35]:
DEFAULT_RELEASE = "gemma-scope-9b-it-res"
#DEFAULT_SAE_ID = "layer_20/width_16k/average_l0_71"
DEFAULT_SAE_ID = "layer_31/width_16k/average_l0_76"
DEFAULT_MODEL = "gemma-2-9b-it"
NEURONPEDIA_FEATURE_API = "https://www.neuronpedia.org/api/feature"

DEVICE = 'cuda'
DTYPE = torch.bfloat16


In [37]:
print("----loading SAE----")
sae = SAE.from_pretrained(
    release=DEFAULT_RELEASE,
    sae_id=DEFAULT_SAE_ID,
    device=DEVICE,
    dtype=DTYPE
)

print("----loading feature catalog----")
if not os.path.exists(f"{DRIVE_DIR}/feature_catalog_{DEFAULT_RELEASE}_{DEFAULT_SAE_ID.replace('/', '_')}.csv"):
    print("\tbuilding feature catalog...")
    feature_catalog = util_neuronpedia.build_feature_catalog()
    feature_catalog.to_csv(f"{DRIVE_DIR}/feature_catalog_{DEFAULT_RELEASE}_{DEFAULT_SAE_ID.replace('/', '_')}.csv", index=False)
else:
    feature_catalog = pd.read_csv(f"{DRIVE_DIR}/feature_catalog_{DEFAULT_RELEASE}_{DEFAULT_SAE_ID.replace('/', '_')}.csv")

print("----loading hooks----")
metadata = getattr(sae.cfg, "metadata", None)
hook_name = getattr(metadata, "hook_name", None)
match = re.search(r"blocks\.(\d+)\.", hook_name)
hook_layer = int(match.group(1)) if match else None

model_name = getattr(metadata, "model_name", None)
prepend_bos = getattr(metadata, "prepend_bos", None)

print(f"\tHook name:\t{hook_name}")
print(f"\tHook layer:\t{hook_layer}")
print(f"\tModel name:\t{model_name}")
print(f"\tPrepend BOS:\t{prepend_bos}")

print("----loading model----")
model = HookedTransformer.from_pretrained_no_processing(model_name,
                                          device=DEVICE, 
                                          dtype=DTYPE, 
                                          default_prepend_bos=prepend_bos, 
                                          #first_n_layers=hook_layer+1
                                          )

----loading SAE----
----loading feature catalog----
	building feature catalog...
----loading hooks----
	Hook name:	blocks.31.hook_resid_post
	Hook layer:	31
	Model name:	gemma-2-9b-it
	Prepend BOS:	True
----loading model----


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-9b-it into HookedTransformer


# Functions

In [38]:
def get_feature_acts(sae, model, hook_name: str, token_ids: torch.Tensor) -> torch.Tensor:
    """Forward a padded token batch through model + SAE.

    Parameters
    ----------
    token_ids : LongTensor [batch, seq_len]  (on model device)

    Returns
    -------
    feature_acts : Tensor [batch, seq_len, num_features]
    """
    with torch.inference_mode():
        _, cache = model.run_with_cache(token_ids, names_filter=[hook_name])
        feature_acts = sae.encode(cache[hook_name])
    return feature_acts

# Data - GSM8K

In [43]:
train_ds, test_ds = util_GSM8k.load_gsm8k()
util_GSM8k.print_rows(train_ds, n=5, label="train")

--- train (7473 rows) ---
[0] {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}
[1] {'question': 'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?', 'answer': 'Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.\nWorking 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.\n#### 10'}
[2] {'question': 'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?', 'answer': "In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.\nBetty's grandparents gave her 15 * 2 

# Dataloader

In [ ]:
train_records = util_GSM8k.build_or_load(train_ds, f"{DRIVE_DIR}/sae_cache/{DEFAULT_RELEASE}_{DEFAULT_SAE_ID.replace('/', '_')}gsm8k_train_acts.pt", model, 512, hook_name, prepend_bos, sae, get_feature_acts)
test_records  = util_GSM8k.build_or_load(test_ds,  f"{DRIVE_DIR}/sae_cache/{DEFAULT_RELEASE}_{DEFAULT_SAE_ID.replace('/', '_')}gsm8k_test_acts.pt", model, 512, hook_name, prepend_bos, sae, get_feature_acts)
train_loader = DataLoader(train_records, batch_size=16, shuffle=False,  drop_last=False,  collate_fn=util_GSM8k.collate_gen)
test_loader  = DataLoader(test_records,  batch_size=16, shuffle=False, drop_last=False, collate_fn=util_GSM8k.collate_gen)

In [ ]:
@torch.no_grad()
def generate_preds(loader, model, sae, hook_name, get_feature_acts,
                   max_new_tokens=256, eot_id=107):
    preds = []
    for batch in tqdm(loader):
        Pmax = batch["input_ids"].shape[1]
        gen = model.generate(batch["input_ids"].to(model.cfg.device),
                             attention_mask=batch["attention_mask"].to(model.cfg.device),
                             max_new_tokens=max_new_tokens, do_sample=False,
                             prepend_bos=False, stop_at_eos=True,
                             eos_token_id=eot_id, verbose=False)
        for j, _id in enumerate(batch["ids"].tolist()):
            prompt = batch["input_ids"][j][batch["attention_mask"][j].bool()]  # unpad this row's prompt
            ans = gen[j, Pmax:]
            cut = (ans == eot_id).nonzero()
            if len(cut): ans = ans[:cut[0,0]]
            full = torch.cat([prompt.to(ans.device), ans]).cpu()
            fa = get_feature_acts(sae, model, hook_name, full[None].to(model.cfg.device))[0].half().cpu()
            preds.append({"id": _id, "pred_input_ids": full,
                          "prompt_len": len(prompt),
                          "feature_acts_withpred": fa.to_sparse()})
    return preds

In [ ]:
def finalize_records(records, loader, model, sae, hook_name, get_feature_acts):
    preds = generate_preds(loader, model, sae, hook_name, get_feature_acts)
    records.sort(key=lambda r: r["id"]); preds.sort(key=lambda p: p["id"])
    for r, p in zip(records, preds):
        assert r["id"] == p["id"]
        r.update({k: p[k] for k in ("pred_input_ids", "feature_acts_withpred")})

    return records

train_records = finalize_records(train_records, train_loader, model, sae, hook_name, get_feature_acts)
test_records  = finalize_records(test_records,  test_loader,  model, sae, hook_name, get_feature_acts)

#torch.save(train_records, train_path)
#torch.save(test_records, test_path)

In [51]:
for i, batch in enumerate(train_loader):
    print(f"Batch {i}:")
    print(f"\tinput_ids shape: {batch['input_ids'].shape}")
    print(f"\tprompt_lens shape: {batch['prompt_lens'].shape}")
    print(f"\tfeature_acts shape: {batch['feature_acts'].shape}")

    r = test_records[0]
    live = get_feature_acts(sae, model, hook_name, r["input_ids"][None])[0].cpu()  # unpadded
    stored = r["feature_acts"].to_dense()
    print(f"\tMax absolute difference: {(live - stored.float()).abs().max()}")
    
    break

Batch 0:
	input_ids shape: torch.Size([16, 225])
	prompt_lens shape: torch.Size([16])
	feature_acts shape: torch.Size([16, 225, 16384])
	Max absolute difference: 0.0


In [31]:
for r in test_records[0:1]:
    ids, pl = r["input_ids"], r["prompt_len"]
    prompt_ids   = ids[:pl].to(DEVICE)      # <-- move to GPU
    true_ans_ids = ids[pl:]

    gen = model.generate(prompt_ids[None], max_new_tokens=256,
                     do_sample=True, prepend_bos=False, verbose=False)
    pred_ans_ids = gen[0, pl:].cpu()        # back to CPU for to_string/printing

    print("=== PROMPT ===\n", model.to_string(prompt_ids.cpu()))
    print("=== TRUE  ===\n",  model.to_string(true_ans_ids))
    print("=== PRED  ===\n",  model.to_string(pred_ans_ids))
    print("=" * 60)

=== PROMPT ===
 <bos>Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
=== TRUE  ===
 Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
=== PRED  ===
 

Find the limit.

$\lim _{x \rightarrow 2} \frac{x^{2}+2 x-8}{x-2} $

Graph each inequality in the coordinate plane.

$4 x + 3y \geq 3 $

Solve the differential equation or initial-value problem that is given.

$\left( y ^ { 2 } + y e ^ { x } \right) \frac { d x } { d y } = 1 , \quad y ( 0 ) = 1 $

Find the solution of the initial value problem. Then draw a phase portrait.

To model the binding of active molecules to protein sites in the liver cell, we write a function $u^{\prime}(t)=-k_{1} u(t)+k_{2} a(t)=F(t)$ which reflects:

a.

In [45]:
def build_it_prompt(question: str) -> str:
    return (f"<start_of_turn>user\n{question}<end_of_turn>\n"
            f"<start_of_turn>model\n")

# end-of-turn token id, used as the stop condition
eot_id = model.tokenizer.convert_tokens_to_ids("<end_of_turn>")

for row in test_ds.select(range(1)):           # use raw dataset, not pre-tokenized records
    prompt_text = build_it_prompt(row["question"])
    prompt_ids = model.to_tokens(prompt_text, prepend_bos=True).to(DEVICE)  # BOS + template
    pl = prompt_ids.shape[1]

    gen = model.generate(prompt_ids, max_new_tokens=256,
                         do_sample=False, prepend_bos=False,
                         stop_at_eos=True, eos_token_id=eot_id, verbose=False)

    pred_ans_ids = gen[0, pl:].cpu()           # only the model's answer

    print("=== PROMPT ===\n", prompt_text)
    print("=== GOLD  ===\n",  row["answer"])
    print("=== PRED  ===\n",  model.to_string(pred_ans_ids))
    print("=" * 60)

    print(model.tokenizer.convert_tokens_to_ids(["<start_of_turn>", "<end_of_turn>"]))
    ids = model.to_tokens(prompt_text, prepend_bos=True)[0]    # drop batch dim
    print([model.tokenizer.decode([int(t)]) for t in ids[:8]])

=== PROMPT ===
 <start_of_turn>user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?<end_of_turn>
<start_of_turn>model

=== GOLD  ===
 Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
=== PRED  ===
 Here's how to solve the problem step-by-step:

1. **Calculate the total eggs used:** Janet uses 3 eggs for breakfast + 4 eggs for muffins = 7 eggs per day.

2. **Calculate the number of eggs sold:** She has 16 eggs - 7 eggs used = 9 eggs to sell.

3. **Calculate her earnings:** She sells 9 eggs * $2 per egg = $18.


**Answer:** Janet makes $18 every day at the farmers' market.<end_of_turn>
[106, 107]
['<bos>', '<start_of_turn>', 'user', '\n', 'Janet', '’', 's', ' ducks']


In [47]:
messages = [{"role": "user", "content": "Write me a poem about Machine Learning."}]

# template -> string -> TL tokens (bare tensor, no dict)
prompt_text = model.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True)
input_ids = model.to_tokens(prompt_text, prepend_bos=False).to("cuda")  # template adds BOS

outputs = model.generate(input_ids, max_new_tokens=256,
                         do_sample=False, prepend_bos=False, verbose=False)
print(model.to_string(outputs[0]))

<bos><start_of_turn>user
Write me a poem about Machine Learning.<end_of_turn>
<start_of_turn>model
In silicon valleys, where data flows deep,
A new intelligence, secrets to keep.
Machine Learning, a mind of its own,
From patterns and numbers, knowledge is sown.

Algorithms dance, a complex ballet,
Learning from examples, come what may.
Neural networks, a web intricate and vast,
Connections forged, insights amassed.

With every iteration, it grows and it learns,
Unveiling truths, as the data it earns.
Predicting the future, classifying with grace,
A digital oracle, in time and space.

From medical diagnoses to self-driving cars,
Machine Learning's reach, extends near and far.
But questions arise, as its power expands,
Of bias and ethics, in these digital lands.

Will it serve humanity, or will it surpass?
A tool for good, or a shadow that casts?
The future unfolds, with each line of code,
Machine Learning's journey, a path yet untrod. 


<end_of_turn><eos>


In [50]:
ids = model.to_tokens(prompt_text, prepend_bos=False)[0]
print((ids == model.tokenizer.bos_token_id).sum().item())   # want exactly 1
print([model.tokenizer.decode([int(t)]) for t in ids])

1
['<bos>', '<start_of_turn>', 'user', '\n', 'Write', ' me', ' a', ' poem', ' about', ' Machine', ' Learning', '.', '<end_of_turn>', '\n', '<start_of_turn>', 'model', '\n']


In [32]:
print(model.cfg.dtype)                     # want torch.bfloat16 for Gemma-2
print(model.cfg.attn_scores_soft_cap, model.cfg.output_logits_soft_cap)
print(model.cfg.model_name)

torch.bfloat16
50.0 30.0
gemma-2-2b


In [26]:
out = model.generate(model.to_tokens("Complete the following syllogism: All humans are mortal. Socrates is a human. Therefore,"),
                     max_new_tokens=100, do_sample=False, verbose=False)
print(model.to_string(out[0]))

<bos>Complete the following syllogism: All humans are mortal. Socrates is a human. Therefore, _____.

The following table shows the number of cell phone subscriptions in the United States for selected years.

a. Find the quadratic function that models the data.

b. Graph the function to determine the year in which the number of cell phone subscriptions is predicted to reach 100 million.

c. Use the function to predict the number of cell phone subscriptions in the United States in 2010.

d. Use the function to predict the year in which the number


# Extra

In [98]:
fa = train_records[0]["feature_acts"]
print(fa._nnz() / fa.numel())          # ≈ 0.004
print(fa._nnz() / fa.shape[0])    

0.009007880028257979
147.58510638297872


In [99]:

nnz_excl_bos = sum((r["feature_acts"].to_dense()[1:] > 0).sum().item() for r in train_records)
tok_excl_bos = sum(r["feature_acts"].shape[0] - 1 for r in train_records)
print(nnz_excl_bos / tok_excl_bos)     # mean L0 ignoring position 0

70.65576404944552


In [97]:
print(os.listdir(f"{REPO_DIR}"))
for file in os.listdir(f"{REPO_DIR}"):
    if file.endswith(".pt"):
        print("file size: ", file, os.path.getsize(f"{REPO_DIR}/{file}") / (1024*1024), "MB")

['gsm8k_test_acts.pt', '.gitignore', '.gitattributes', 'sae_cache', 'util_nanogpt.py', '__pycache__', 'sae_model.ipynb', '.git', 'feature_catalog.csv', 'util_GSM8k.py', 'util_neuronpedia.py', 'gsm8k_train_acts.pt']
file size:  gsm8k_test_acts.pt 459.74370765686035 MB
file size:  gsm8k_train_acts.pt 2566.7798233032227 MB


# Get Activations

In [ ]:
train_dataloader, test_dataloader = util_GSM8k.make_dataloaders(train_ds, test_ds, model, prepend_bos=True, batch_size=16, max_length=512)

In [93]:
def get_feature_acts(sae, model, hook_name: str, token_ids: torch.Tensor) -> torch.Tensor:
    """Forward a padded token batch through model + SAE.

    Parameters
    ----------
    token_ids : LongTensor [batch, seq_len]  (on model device)

    Returns
    -------
    feature_acts : Tensor [batch, seq_len, num_features]
    """
    with torch.inference_mode():
        _, cache = model.run_with_cache(token_ids, names_filter=[hook_name])
        feature_acts = sae.encode(cache[hook_name])
    return feature_acts


In [62]:
for i, batch in enumerate(train_dataloader):
    print(f"batch {i}:")
    print(f"\tinput_ids shape: {batch['input_ids'].shape}")
    print(f"\tprompt_lens shape: {batch['prompt_lens'].shape}")

    print(batch['input_ids'])
    print(batch['prompt_lens'])

    print("Feature Acts")
    feature_acts = get_feature_acts(sae, model, hook_name, batch['input_ids'])
    print(f"\tfeature_acts shape: {feature_acts.shape}")
    print(feature_acts)

    break

batch 0:
	input_ids shape: torch.Size([16, 285])
	prompt_lens shape: torch.Size([16])
tensor([[     0,      0,      0,  ..., 235248, 235304, 235304],
        [     0,      0,      0,  ..., 235284, 235315, 235318],
        [     0,      0,      0,  ..., 235284, 235310, 235276],
        ...,
        [     0,      0,      0,  ..., 235284, 235308, 235276],
        [     0,      0,      0,  ..., 235248, 235308, 235308],
        [     0,      0,      0,  ..., 235274, 235324, 235315]])
tensor([215,  78, 223, 164, 111, 140, 193, 174, 142,  91, 131, 198, 136, 164,
        234, 211])
Feature Acts
	feature_acts shape: torch.Size([16, 285, 16384])
tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0.

In [76]:
for i, batch in enumerate(train_dataloader):
    print(f"batch {i}:")
    print(f"\tinput_ids shape: {batch['input_ids'].shape}")
    print(f"\tprompt_lens shape: {batch['prompt_lens'].shape}")

    feature_acts = get_feature_acts(sae, model, hook_name, batch['input_ids'])
    print(f"\tfeature_acts shape: {feature_acts.shape}")

    break

batch 0:
	input_ids shape: torch.Size([16, 267])
	prompt_lens shape: torch.Size([16])
	feature_acts shape: torch.Size([16, 267, 16384])


### Github

In [132]:
!git -C {REPO_DIR} pull

import importlib
importlib.reload(util_GSM8k)

import util_neuronpedia
import util_nanogpt
import util_GSM8k

Already up to date.


In [15]:
!git -C {REPO_DIR} add .
!git -C {REPO_DIR} commit -m "sae_cache"
!git -C {REPO_DIR} push origin main

^C
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	__pycache__/
	sae_cache/

nothing added to commit but untracked files present (use "git add" to track)
Everything up-to-date


In [18]:
for file in os.listdir(f"{REPO_DIR}/sae_cache"):
    print(file)
    print(f"file size: {os.path.getsize(os.path.join(REPO_DIR, 'sae_cache', file)) / (1024**3)} gigabytes")

gsm8k_test_features.pt
file size: 4.952468877658248 gigabytes
gsm8k_train_features.pt
file size: 27.309960260987282 gigabytes


In [ ]:
"""
def get_feature_sequence_old(sae, model, prepend_bos, hook_name, top_k, prompt):
    with torch.inference_mode():
        tokens = model.to_tokens(prompt, prepend_bos=prepend_bos)
        str_tokens = model.to_str_tokens(tokens[0])

        _, cache = model.run_with_cache(tokens, names_filter=[hook_name])
        activations = cache[hook_name]
        feature_acts = sae.encode(activations)

    return feature_acts, str_tokens
"""